[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/minimind-colab/blob/main/notebooks/04_SFT_and_LoRA.ipynb)

# Module 4: Supervised Fine-Tuning & LoRA

In [ ]:
# @title 🔧 Environment Setup (Run this first!)
import os

gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU"
print(f"GPU: {gpu_info[0]}")

if not os.path.exists('minimind-colab'):
    !git clone https://github.com/Boyu-Zhang-UOI/minimind-colab.git
os.chdir('minimind-colab')

!pip install -q transformers==4.46.3 datasets==3.6.0 jinja2==3.1.2 tokenizers

import sys
sys.path.insert(0, '.')
print("✅ Setup complete!")

## 📚 Overview

After pretraining, a model can predict the next token but cannot follow instructions.
**Supervised Fine-Tuning (SFT)** teaches it to respond to user prompts in a conversational format.

**What we will cover:**
1. SFT data format — multi-turn conversations with roles
2. Chat templates — structuring conversations with special tokens
3. Label masking — training only on assistant responses
4. Full SFT training loop
5. LoRA — parameter-efficient fine-tuning
6. Merging LoRA weights back into the base model

**Key source files:** `dataset/lm_dataset.py` (SFTDataset), `trainer/train_full_sft.py`, `model/model_lora.py`, `trainer/train_lora.py`

## 1. SFT Data Format

SFT data consists of **multi-turn conversations** in JSONL format:

```json
{
  "conversations": [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is machine learning?"},
    {"role": "assistant", "content": "Machine learning is a branch of AI..."}
  ]
}
```

Each conversation has:
- **system** (optional): Sets the assistant's behavior
- **user**: The human's message
- **assistant**: The model's response (this is what we train on)

In [ ]:
import torch
import sys
sys.path.insert(0, '.')
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('./model')

# A multi-turn conversation
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "The answer is 4."},
    {"role": "user", "content": "And 3+3?"},
    {"role": "assistant", "content": "The answer is 6."}
]

# Apply chat template
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print("=== Formatted Conversation ===")
print(formatted)
print(f"\n=== Token count: {len(tokenizer.encode(formatted))} ===")

## 2. Label Masking — The Key Idea of SFT

In SFT, we **only compute loss on assistant responses**, not on user prompts or system messages.

Why? The model should learn to *generate* responses, not to *memorize* prompts.

**Implementation:**
- Labels are a copy of `input_ids`
- All tokens that are NOT part of assistant responses are set to `-100`
- PyTorch's `cross_entropy` with `ignore_index=-100` skips these positions

```
<|im_start|>user    ← labels = -100 (masked)
What is 2+2?        ← labels = -100 (masked)
<|im_end|>          ← labels = -100 (masked)
<|im_start|>assistant
The answer is 4.    ← labels = actual token IDs (trained!)
<|im_end|>          ← labels = actual token ID (trained!)
```

In [ ]:
from dataset.lm_dataset import SFTDataset
import json

# Create a small SFT dataset
sft_data = [
    {"conversations": [
        {"role": "user", "content": "What is AI?"},
        {"role": "assistant", "content": "AI stands for Artificial Intelligence."}
    ]},
    {"conversations": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain gravity."},
        {"role": "assistant", "content": "Gravity is a fundamental force."},
        {"role": "user", "content": "More details?"},
        {"role": "assistant", "content": "It attracts objects with mass toward each other."}
    ]},
]

data_path = '/tmp/sample_sft.jsonl'
with open(data_path, 'w') as f:
    for item in sft_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

dataset = SFTDataset(data_path, tokenizer, max_length=128)
input_ids, labels = dataset[1]  # Multi-turn example

print("=== Label Masking Visualization ===")
print(f"{'Pos':>4} {'Token':>20} {'Label':>8}  Status")
print("-" * 55)
for i in range(min(60, len(input_ids))):
    token_text = tokenizer.decode([input_ids[i].item()])
    label_val = labels[i].item()
    status = "📝 TRAIN" if label_val != -100 else "🚫 MASK"
    label_str = f"{label_val}" if label_val != -100 else "-100"
    print(f"{i:4d} {token_text!r:>20} {label_str:>8}  {status}")

## 3. Full SFT Training

The SFT training loop is nearly identical to pretraining, with key differences:

| Aspect | Pretraining | SFT |
|--------|------------|-----|
| Data | Raw text | Conversations |
| Learning rate | 5e-4 | 1e-5 |
| Sequence length | 340 | 768 |
| Labels | All tokens | Assistant only |
| Base weights | Random / none | Pretrained |

> **Note:** In this demo we use `max_length=128` for speed. In production, SFT typically uses 768+.

In [ ]:
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from torch import optim
from torch.utils.data import DataLoader
from contextlib import nullcontext

config = MiniMindConfig(hidden_size=768, num_hidden_layers=8)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = MiniMindForCausalLM(config).to(device)

dataset = SFTDataset(data_path, tokenizer, max_length=128)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
dtype = torch.bfloat16 if device == 'cuda' and torch.cuda.is_bf16_supported() else torch.float32
autocast_ctx = torch.cuda.amp.autocast(dtype=dtype) if device == 'cuda' else nullcontext()
scaler = torch.cuda.amp.GradScaler(enabled=(dtype == torch.float16))

model.train()
num_epochs = 5
losses = []

print(f"SFT Training: {num_epochs} epochs, {len(loader)} steps/epoch")
print(f"{'='*50}")

for epoch in range(num_epochs):
    epoch_loss = 0
    for step, (input_ids, labels) in enumerate(loader, 1):
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        with autocast_ctx:
            res = model(input_ids, labels=labels)
            loss = res.loss + res.aux_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        current_loss = loss.item()
        epoch_loss += current_loss
        losses.append(current_loss)

    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss/len(loader):.4f}")

print(f"\n✅ SFT complete! Final loss: {losses[-1]:.4f}")

## 4. LoRA — Low-Rank Adaptation

LoRA is a **parameter-efficient** fine-tuning method that adds small trainable matrices to the frozen base model:

$$W' = W + \Delta W = W + A \cdot B$$

Where:
- $W$ is the original weight (frozen, not updated)
- $A$ is shape `(in_features, rank)` — initialized from $\mathcal{N}(0, 0.02)$
- $B$ is shape `(rank, out_features)` — initialized to **zeros**
- At initialization: $\Delta W = A \cdot B = 0$ (model starts unchanged)

**Why LoRA works:**
- Rank 16 captures most meaningful weight updates
- Only ~5% of parameters are trainable
- Checkpoint is ~5MB instead of ~300MB
- Multiple LoRA adapters can be swapped on the same base model

In [ ]:
from model.model_lora import LoRA, apply_lora, save_lora, merge_lora
import torch.nn as nn

# Inspect the LoRA class
lora = LoRA(in_features=768, out_features=768, rank=16)
print("=== LoRA Structure ===")
print(f"Matrix A: {lora.A.weight.shape}  (in → rank)")
print(f"Matrix B: {lora.B.weight.shape}  (rank → out)")
print(f"LoRA params: {sum(p.numel() for p in lora.parameters()):,}")
print(f"Full Linear params: {768*768:,}")
print(f"LoRA/Full ratio: {sum(p.numel() for p in lora.parameters())/(768*768)*100:.2f}%")

# Verify initialization: A ~ N(0, 0.02), B = 0
print(f"\nA weight stats: mean={lora.A.weight.data.mean():.4f}, std={lora.A.weight.data.std():.4f}")
print(f"B weight stats: mean={lora.B.weight.data.mean():.4f}, std={lora.B.weight.data.std():.4f}")
print(f"Initial ΔW = A·B output norm: {lora(torch.randn(1, 768)).norm():.6f} (should be ~0)")

In [ ]:
# Create a fresh model and apply LoRA
model_lora = MiniMindForCausalLM(config).to(device)
apply_lora(model_lora, rank=16)

# Count parameters
total_params = sum(p.numel() for p in model_lora.parameters())
lora_params = sum(p.numel() for n, p in model_lora.named_parameters() if 'lora' in n)
base_params = total_params - lora_params

print("=== After Applying LoRA ===")
print(f"Total params:  {total_params:,}")
print(f"Base params:   {base_params:,} (frozen)")
print(f"LoRA params:   {lora_params:,} (trainable)")
print(f"LoRA fraction: {lora_params/total_params*100:.2f}%")

# Freeze non-LoRA parameters
trainable_params = []
for name, param in model_lora.named_parameters():
    if 'lora' in name:
        param.requires_grad = True
        trainable_params.append(param)
    else:
        param.requires_grad = False

print(f"\nTrainable params: {sum(p.numel() for p in trainable_params):,}")
print(f"Frozen params: {sum(p.numel() for p in model_lora.parameters() if not p.requires_grad):,}")

In [ ]:
# LoRA training loop
optimizer_lora = optim.AdamW(trainable_params, lr=1e-4)
lora_losses = []

model_lora.train()
print("LoRA Training (higher LR since fewer params):")
print(f"{'='*50}")

for epoch in range(5):
    epoch_loss = 0
    for step, (input_ids, labels) in enumerate(loader, 1):
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        with autocast_ctx:
            res = model_lora(input_ids, labels=labels)
            loss = res.loss + res.aux_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer_lora.step()
        optimizer_lora.zero_grad(set_to_none=True)

        current_loss = loss.item()
        epoch_loss += current_loss
        lora_losses.append(current_loss)

    print(f"Epoch {epoch+1}/5 | Loss: {epoch_loss/len(loader):.4f}")

print(f"\n✅ LoRA training complete!")

## 5. Saving and Merging LoRA

LoRA supports three operations:

1. **`save_lora(model, path)`** — Save only LoRA weights (~5MB)
2. **`load_lora(model, path)`** — Load LoRA weights onto a base model
3. **`merge_lora(model, lora_path, save_path)`** — Fuse $A \cdot B$ into $W$ and save full model

After merging, the model runs at full speed with no LoRA overhead.

In [ ]:
import os

# Save LoRA weights only
os.makedirs('/tmp/lora_out', exist_ok=True)
save_lora(model_lora, '/tmp/lora_out/lora_demo_768.pth')

lora_size = os.path.getsize('/tmp/lora_out/lora_demo_768.pth')
print(f"LoRA checkpoint size: {lora_size/1024:.1f} KB ({lora_size/1024/1024:.2f} MB)")

# For comparison, save full model
torch.save(model_lora.state_dict(), '/tmp/lora_out/full_model.pth')
full_size = os.path.getsize('/tmp/lora_out/full_model.pth')
print(f"Full model size:      {full_size/1024/1024:.1f} MB")
print(f"Size ratio: {lora_size/full_size*100:.1f}%")

## ✏️ Exercises

1. **Inspect label masking**: For the multi-turn conversation sample, count how many tokens are trained vs masked. What percentage of tokens are actually used for the loss?

2. **LoRA rank experiment**: Try rank=4, rank=16, and rank=64. How does the number of trainable parameters change? Does higher rank always mean better quality?

3. **Compare SFT vs LoRA**: After training both models for the same number of steps, compare their generation quality on the same prompts. Which produces more coherent responses?

4. **Domain-specific LoRA**: Create a small dataset of conversations about a specific topic (e.g., cooking, sports). Train a LoRA adapter on it and test whether the model becomes more knowledgeable in that domain.

## 📝 Summary

In this module we covered:

- **SFT data format** — Multi-turn conversations with system/user/assistant roles
- **Chat templates** — Structuring input with `<|im_start|>`/`<|im_end|>` markers
- **Label masking** — Training only on assistant responses (labels = -100 for everything else)
- **Full SFT** — Fine-tuning all parameters with lower learning rate
- **LoRA** — Low-rank adaptation: $W' = W + A \cdot B$ with rank=16
- **Parameter efficiency** — ~5% trainable params, ~5MB checkpoints
- **Save/merge** — LoRA weights can be saved, loaded, and merged into the base model

🔜 **Next module:** Alignment with DPO and GRPO — teaching the model to prefer better responses!